<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="20%">
</div>

<br>

# REGULARIZATION AND CROSS-VALIDATION USING LUDWIG

<br>

**About:** This tutorial introduces regularization and cross-validation in the context of NLP using Ludwig, a declarative deep learning framework. Starting from a baseline sentiment classifier, you will apply early stopping and learning rate control to manage overfitting, then compare encoder architectures as a practical form of model cross-validation - all without writing a custom training loop.

**Learning Goals:**
- Understand what regularization means in deep learning and how early stopping prevents overfitting
- Apply learning rate tuning as a complementary regularization lever
- Compare encoder architectures (RNN, Parallel CNN, CNNRNN) on the same dataset as a cross-validation strategy
- Load and preprocess text data from the Stanford Sentiment Treebank using Torchtext
- Define Ludwig model configurations as Python dictionaries and train with a single API call
- Interpret training and validation loss curves to diagnose overfitting versus underfitting

**Keywords:** regularization, cross-validation, early stopping, Ludwig, NLP, sentiment analysis

**Prerequisite Knowledge:** (1) Python, (2) basic Machine Learning, (3) Natural Language Processing fundamentals

**Target User:** Data scientists, applied machine learning engineers, and developers

<hr style="border: 4px solid#003262;" />

<a name="Part_table_contents" id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 0: PREREQUISITE KNOWLEDGE](#Part_0)
> #### [PART 1: THE STANFORD SENTIMENT TREEBANK](#Part_1)
> #### [PART 2: REGULARIZATION - EARLY STOPPING AND LEARNING RATE](#Part_2)
> #### [PART 3: CROSS-VALIDATION VIA ENCODER ARCHITECTURE COMPARISON](#Part_3)
> #### [PART 4: VISUALIZING AND EVALUATING RESULTS](#Part_4)

#### APPENDIX

> #### [APPENDIX: INSTALLATION AND SETUP](#Appendix_1)

<br>

<a id="Part_0"></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **PREREQUISITE** KNOWLEDGE

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tf_logo_social.png" align="center" width="20%" padding="10"><br>
    <br>
</div>

This tutorial assumes a working knowledge of Python and basic machine learning. Familiarity with deep learning and NLP will make the encoder comparisons in Parts 2 and 3 easier to follow, but is not required to run the code.

<br>

**PYTHON**

<details>
<summary>Expand</summary>

Ludwig is built using [Python](https://www.python.org/) and relies on common Python packages for operation. If you have strong programming experience in another language you will likely be fine; otherwise:

- [**Python (EDX free)**](https://www.edx.org/course?search_query=python)
- [**Python (Coursera free)**](https://www.coursera.org/search?query=python)
</details>

<br>

**MACHINE LEARNING**

<details>
<summary>Expand</summary>

Regularization and cross-validation are standard techniques for improving model generalization. Having exposure to concepts like loss functions, gradient descent, and train/validation/test splits will make Parts 2 and 3 more intuitive:

- [**A Course in Machine Learning (ebook)**](http://ciml.info/)
- [**An Introduction to Statistical Learning (book)**](https://www.statlearning.com/)
</details>

<br>

**DEEP LEARNING**

<details>
<summary>Expand</summary>

Understanding how neural networks learn - forward pass, loss computation, backpropagation - will help you interpret the learning curves in Part 4:

- [**Deep Learning (MIT Book)**](http://www.deeplearningbook.org/)
- **Dive into Deep Learning** [ebook](https://d2l.ai/d2l-en.pdf) | [book website](http://d2l.ai/)
</details>

<br>

**NATURAL LANGUAGE PROCESSING (NLP)**

<details>
<summary>Expand</summary>

This tutorial uses a text classification dataset (Stanford Sentiment Treebank). Knowing what tokenization, word embeddings, and sequence classification mean will help, but is not a prerequisite for running the code:

- [**NLP Online Courses (Coursera)**](https://www.coursera.org/search?query=nlp)
- [**NLP Class Website (Stanford CS224n)**](https://web.stanford.edu/class/cs224n/)
</details>

<br>

**SENTIMENT ANALYSIS**

<details>
<summary>Expand</summary>

[**Sentiment analysis**](https://en.wikipedia.org/wiki/Sentiment_analysis) is a subfield of Natural Language Understanding that applies machine learning to assign a positive, negative, or neutral orientation to text. In this tutorial sentiment analysis is the application domain - the techniques being taught (regularization, cross-validation) transfer to any classification problem.
</details>

___

**Setup:** Run the cell below to import all packages before proceeding. See the Appendix for installation instructions.

In [ ]:
import os
import logging
from pprint import pprint

import pandas as pd
from torchtext import data as torchtext_data        # torchtext==0.6.0
from torchtext import datasets                       # torchtext==0.6.0
from nltk.tokenize.treebank import TreebankWordDetokenizer
from tqdm import trange

import ludwig
from ludwig.api import LudwigModel                  # ludwig==0.2.1
from IPython.display import Image as im

In [ ]:
import tensorflow as tf

device_name = tf.test.gpu_device_name()
if device_name == '/device:GPU:0':
    print(f'GPU found: {device_name}')
else:
    print('WARNING: no GPU found. Training will run on CPU and may be very slow.')
    print('A GPU is strongly recommended for the CNNRNN and BERT cells.')

<a id="Part_1"></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## THE **STANFORD SENTIMENT TREEBANK**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tensorflow_thumbnail-01.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

The [**Stanford Sentiment Treebank (SST)**](https://nlp.stanford.edu/sentiment/) is a sentence-level corpus of 10,662 movie review sentences, each assigned one of five fine-grained sentiment labels: *very positive*, *positive*, *neutral*, *negative*, or *very negative*. It was introduced by [Socher et al. (2013)](https://www.researchgate.net/publication/284039049_Recursive_deep_models_for_semantic_compositionality_over_a_sentiment_treebank) as a benchmark designed to expose the limitations of sentiment models that treat a sentence as a bag of words.

What makes SST unusual is that it provides labels not just at the sentence level but for **every phrase** in each sentence's parse tree. A sentence of 15 words may generate dozens of labeled sub-phrases. This matters for training: setting `train_subtrees=True` includes all subtree-level examples in the training set, significantly expanding the effective training data without collecting new sentences. The validation and test sets use only the sentence-level labels.

We use Torchtext to load SST, split it into training, validation, and test sets, and convert the tokenized sequences back to strings using `TreebankWordDetokenizer` - the format Ludwig expects for text features. Labels are mapped to integer indices (0-4) via `class2idx`.

___

**Note:** The Torchtext SST loader and field API below are verified against torchtext 0.6.0 (2020-03-01). The `torchtext.data.Field` and `datasets.SST.splits()` interface changed substantially in torchtext 0.9+. Confirm at [PyTorch Text docs](https://pytorch.org/text/) before upgrading.

___

*Sources consulted:*
- *Socher, R. et al. (2013). Recursive Deep Models for Semantic Compositionality over a Sentiment Treebank. EMNLP.*
- *Stanford NLP Group. Stanford Sentiment Treebank. https://nlp.stanford.edu/sentiment/*
- *PyTorch. Torchtext 0.6.0 documentation.*

In [ ]:
# Map integer indices to class names
idx2class = ['positive', 'negative', 'very positive', 'very negative', 'neutral']
class2idx = {c: i for i, c in enumerate(idx2class)}

# Initialize Torchtext Field objects
text_field = torchtext_data.Field()
label_field = torchtext_data.Field(sequential=False)

# Load SST with subtree-level training examples; val/test use sentence-level only
train_data, val_data, test_data = datasets.SST.splits(
    text_field,
    label_field,
    fine_grained=True,
    train_subtrees=True,
)

# SST text is already tokenized; detokenize to single strings for Ludwig
detokenizer = TreebankWordDetokenizer()

x_train, y_train = [], []
for i in trange(len(train_data), desc='train', ascii=True):
    x_train.append(detokenizer.detokenize(vars(train_data[i])['text']))
    y_train.append(class2idx[vars(train_data[i])['label']])

x_val, y_val = [], []
for i in trange(len(val_data), desc='val', ascii=True):
    x_val.append(detokenizer.detokenize(vars(val_data[i])['text']))
    y_val.append(class2idx[vars(val_data[i])['label']])

x_test, y_test = [], []
for i in trange(len(test_data), desc='test', ascii=True):
    x_test.append(detokenizer.detokenize(vars(test_data[i])['text']))
    y_test.append(class2idx[vars(test_data[i])['label']])

train_df = pd.DataFrame({'text': x_train, 'label': y_train})
val_df   = pd.DataFrame({'text': x_val,   'label': y_val})
test_df  = pd.DataFrame({'text': x_test,  'label': y_test})

In [ ]:
print(f'Training examples: {len(train_df)}')
print(f'Validation examples: {len(val_df)}')
print(f'Test examples: {len(test_df)}')

pd.set_option('display.max_colwidth', None)  # use -1 for pandas < 1.0
train_df.head()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->

> **The data loading cell above uses `train_subtrees=True`. Run the cell below to compare the training set size with and without subtree annotations. Then write a 2-3 sentence explanation (as a comment) of why a larger training set generally helps when comparing encoder architectures.**

<br>

```python
text_field_check = torchtext_data.Field()
label_field_check = torchtext_data.Field(sequential=False)
train_no_subtrees, _, _ = datasets.SST.splits(
    text_field_check, label_field_check,
    fine_grained=True, train_subtrees=False
)
print(f'Training examples with subtrees:    {len(train_df)}')
print(f'Training examples without subtrees: {len(train_no_subtrees)}')
# Why does a larger training set help when comparing architectures?
# Your explanation here:
```

<hr style="border: 2px solid#003262;" />

In [ ]:
text_field_check = torchtext_data.Field()
label_field_check = torchtext_data.Field(sequential=False)
train_no_subtrees, _, _ = datasets.SST.splits(
    text_field_check, label_field_check,
    fine_grained=True, train_subtrees=False
)
print(f'Training examples with subtrees:    {len(train_df)}')
print(f'Training examples without subtrees: {len(train_no_subtrees)}')

# Why does a larger training set help when comparing architectures?
# Your explanation here:


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id="Part_2"></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## REGULARIZATION - **EARLY STOPPING** AND LEARNING RATE

<div align="center" style="font-size:12px; font-family:FreeMono; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tF_update-03.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

**Overfitting** is the central failure mode in supervised learning: a model memorizes the training data well enough to minimize training loss but fails to generalize to new examples. In deep learning, overfitting appears as a divergence between training and validation loss - training loss continues to fall while validation loss plateaus or rises. **Regularization** is any technique that reduces this tendency.

Ludwig exposes two regularization levers in the `training` block of a model configuration dictionary:

<br>

**Early Stopping** (`early_stop: N`)

Ludwig monitors validation loss after each epoch. If the validation loss fails to improve for N consecutive epochs, training halts. This works because validation loss tracks generalization: once it stops improving, additional training steps are more likely to narrow the gap to training loss through memorization than to genuinely improve the model.

Setting `early_stop: 100` (used in the baseline below) is permissive - the model may plateau for a long time before stopping. A smaller value such as `early_stop: 10` applies stricter regularization: training ends sooner, reducing the risk of overfitting but also risking stopping before the model has converged past a temporary plateau.

Early stopping is not just a training trick - it is a genuine regularization technique in the sense of Goodfellow et al. (2016, Chapter 7): it constrains the effective capacity of the hypothesis space by capping the number of gradient steps, and its regularization effect increases with smaller `early_stop` values.

<br>

**Learning Rate** (`learning_rate: r`)

The learning rate controls the magnitude of each gradient-descent update. A rate that is too high causes training loss to oscillate or diverge - the optimizer overshoots the loss minimum on each step. A rate that is too low causes slow convergence or premature plateauing in a shallow local minimum. Both appear in learning curves: oscillation as jagged loss curves, underfitting as flat or slowly declining loss.

The baseline below uses Ludwig's default learning rate (no explicit setting). In Part 3, different encoder architectures will use explicitly lower rates to stabilize convolutional weight updates early in training.

___

**Note:** Ludwig 0.2.1 training configuration keys (`early_stop`, `learning_rate`) are verified against the Ludwig 0.2.1 source. These keys and their defaults changed in Ludwig 0.6+. See [Ludwig changelog](https://github.com/ludwig-ai/ludwig/blob/master/CHANGELOG.md) before upgrading.

___

*Sources consulted:*
- *Goodfellow, I., Bengio, Y., Courville, A. (2016). Deep Learning, Chapter 7: Regularization. MIT Press.*
- *Prechelt, L. (1998). Early Stopping - But When? In Neural Networks: Tricks of the Trade. Springer LNCS 1524.*
- *Krogh, A., Hertz, J.A. (1992). A simple weight decay can improve generalization. NIPS 4.*

In [ ]:
# Baseline: vanilla RNN encoder with early stopping
# early_stop: 100 means training halts after 100 consecutive epochs without validation improvement
model_definition_rnn = {
    'input_features': [
        {'name': 'text', 'type': 'text', 'level': 'word', 'encoder': 'rnn'},
    ],
    'output_features': [
        {'name': 'label', 'type': 'category'}
    ],
    'training': {
        'early_stop': 100
    }
}

print('Creating baseline RNN model...')
model_rnn = LudwigModel(model_definition_rnn, logging_level=logging.WARNING)

print('Training...')
model_rnn.train(
    data_train_df=train_df,
    data_validation_df=val_df,
    data_test_df=test_df
)

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->

> **The baseline uses `early_stop: 100`. In the cell below, define a second model configuration using `early_stop: 5`. Before running it, write a comment predicting: will this model train for more or fewer epochs than the baseline? Will it overfit more or less? Explain why in 2-3 sentences.**

<br>

```python
model_definition_strict = {
    'input_features': [
        {'name': 'text', 'type': 'text', 'level': 'word', 'encoder': 'rnn'},
    ],
    'output_features': [
        {'name': 'label', 'type': 'category'}
    ],
    'training': {
        'early_stop': ...  # TODO: set to 5
    }
}
# Prediction - expected training duration vs baseline (more/fewer epochs):
# Prediction - expected overfitting behavior:
# Why:
```

<hr style="border: 2px solid#003262;" />

In [ ]:
model_definition_strict = {
    'input_features': [
        {'name': 'text', 'type': 'text', 'level': 'word', 'encoder': 'rnn'},
    ],
    'output_features': [
        {'name': 'label', 'type': 'category'}
    ],
    'training': {
        'early_stop': ...  # TODO: set to 5
    }
}

# Prediction - expected training duration vs baseline (more/fewer epochs):
# Prediction - expected overfitting behavior:
# Why:


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id="Part_3"></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## CROSS-VALIDATION VIA **ENCODER ARCHITECTURE** COMPARISON

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tF_update-05.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

**Cross-validation** is a strategy for estimating how well a model will generalize to new data. In formal k-fold cross-validation, a dataset is divided into k subsets: one fold is held out as the validation set while the model trains on the remaining k-1 folds, rotating until every fold has served as the validation set. Performance is averaged across all k runs.

This tutorial uses a simpler, computationally cheaper form: **model selection cross-validation**. We train three encoder architectures on the same fixed train/validation/test split and compare their test-set accuracy. This answers the question: *which architectural assumption about how text encodes sentiment generalizes best to unseen sentences?*

Each encoder embodies a different **inductive bias** - a built-in assumption about which patterns in the input are most informative:

<br>

**Vanilla RNN** (baseline, Part 2)
Processes text sequentially, token by token. The hidden state at each step carries a summary of all tokens seen so far. Inductive bias: global token order and long-range dependencies matter for sentiment. This is reasonable - negation constructions like "not particularly good" require holding context across multiple tokens.

<br>

**Parallel CNN** (`encoder: parallel_cnn`)
Applies convolutional filters across sliding windows of tokens, extracting local n-gram features. Multiple filter widths run in parallel, capturing patterns at different scales (bigrams, trigrams, etc.). Inductive bias: short, local phrase patterns are the most discriminative features for sentiment. Order within a window matters; global sentence order is less important. Applied with `learning_rate: 0.00001` - convolutional weights trained on text embeddings are sensitive to large gradient updates early in training; a lower rate gives filters time to specialize before the output layer's loss gradient dominates.

<br>

**CNNRNN** (`encoder: cnnrnn`)
Stacks CNN layers (local feature extraction) followed by an RNN layer (sequential aggregation of local features across the full sequence). Inductive bias: local n-gram patterns exist *and* their sequential arrangement across the sentence carries additional information. Combines both prior encoders' strengths, but is also the most computationally expensive. Applied with `learning_rate: 0.00001` for the same reason as Parallel CNN.

___

**Note:** Ludwig encoder names (`rnn`, `parallel_cnn`, `cnnrnn`) are verified against Ludwig 0.2.1. These names changed or were reorganized in Ludwig 0.6+.

___

*Sources consulted:*
- *Kohavi, R. (1995). A study of cross-validation and bootstrap for accuracy estimation and model selection. IJCAI, 14(2), 1137-1145.*
- *Kim, Y. (2014). Convolutional Neural Networks for Sentence Classification. EMNLP. arXiv:1408.5882.*
- *Goodfellow, I., Bengio, Y., Courville, A. (2016). Deep Learning, Chapter 6. MIT Press.*

In [ ]:
# Parallel CNN: convolutional filters capture local n-gram patterns
# Lower learning rate stabilizes filter training on text embeddings
model_definition_cnn = {
    'input_features': [
        {'name': 'text', 'type': 'text', 'level': 'word', 'encoder': 'parallel_cnn'},
    ],
    'output_features': [
        {'name': 'label', 'type': 'category'}
    ],
    'training': {
        'learning_rate': 0.00001,
        'early_stop': 10,
    }
}

print('Creating Parallel CNN model...')
model_cnn = LudwigModel(model_definition_cnn, logging_level=logging.WARNING)

print('Training...')
model_cnn.train(
    data_train_df=train_df,
    data_validation_df=val_df,
    data_test_df=test_df
)

In [ ]:
# CNNRNN: CNN extracts local patterns, RNN aggregates them sequentially
model_definition_cnnrnn = {
    'input_features': [
        {'name': 'text', 'type': 'text', 'level': 'word', 'encoder': 'cnnrnn'},
    ],
    'output_features': [
        {'name': 'label', 'type': 'category'}
    ],
    'training': {
        'learning_rate': 0.00001,
        'early_stop': 10,
    }
}

print('Creating CNNRNN model...')
model_cnnrnn = LudwigModel(model_definition_cnnrnn, logging_level=logging.WARNING)

print('Training...')
model_cnnrnn.train(
    data_train_df=train_df,
    data_validation_df=val_df,
    data_test_df=test_df
)

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->

> **The Parallel CNN uses `learning_rate: 0.00001` while the baseline RNN used Ludwig's default (a higher value). In the cell below, write a comment predicting what would happen if you ran the CNN with `learning_rate: 0.001` instead. Then optionally uncomment the training block and verify your prediction against the learning curves in Part 4.**

<br>

```python
# Prediction: what happens to CNN training with learning_rate=0.001?
# Your explanation (2-3 sentences):

model_definition_experiment = {
    'input_features': [
        {'name': 'text', 'type': 'text', 'level': 'word', 'encoder': 'parallel_cnn'},
    ],
    'output_features': [{'name': 'label', 'type': 'category'}],
    'training': {'learning_rate': 0.001, 'early_stop': 10}
}
# model_exp = LudwigModel(model_definition_experiment, logging_level=logging.WARNING)
# model_exp.train(data_train_df=train_df, data_validation_df=val_df, data_test_df=test_df)
```

<hr style="border: 2px solid#003262;" />

In [ ]:
# Prediction: what happens to CNN training with learning_rate=0.001?
# Your explanation (2-3 sentences):

model_definition_experiment = {
    'input_features': [
        {'name': 'text', 'type': 'text', 'level': 'word', 'encoder': 'parallel_cnn'},
    ],
    'output_features': [{'name': 'label', 'type': 'category'}],
    'training': {'learning_rate': 0.001, 'early_stop': 10}
}
# Uncomment to run:
# model_exp = LudwigModel(model_definition_experiment, logging_level=logging.WARNING)
# model_exp.train(data_train_df=train_df, data_validation_df=val_df, data_test_df=test_df)

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id="Part_4"></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **VISUALIZING** AND EVALUATING RESULTS

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/tensorboard_logo_social.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

A **learning curve** plots training loss and validation loss across training epochs. Reading these curves is the primary diagnostic tool for understanding what happened during training.

<br>

**Healthy training** - both losses decrease steadily early in training, then converge and plateau at similar values. The gap between them (the generalization gap) stays small throughout.

**Overfitting** - training loss continues to fall while validation loss plateaus or rises. The widening generalization gap is the signature: the model is learning patterns specific to training data that do not transfer to validation data. Remedies include reducing `early_stop` patience, applying dropout within the encoder, or using a simpler encoder architecture.

**Underfitting** - both losses remain high and plateau early. The model lacks capacity, or the learning rate is too low for convergence. Remedies include a more expressive encoder, a higher learning rate, or more training epochs.

<br>

Ludwig saves training statistics to a directory under `results/` after each training run. The `ludwig visualize` command reads those statistics and generates learning curve PNGs.

___

**Note:** Ludwig 0.2.1 names the results directory `api_experiment_run` for the first run via `LudwigModel.train()`. Subsequent runs append `_0`, `_1`, etc. Check `results/` after training to confirm the directory name before running the visualization cell.

___

*Sources consulted:*
- *Goodfellow, I., Bengio, Y., Courville, A. (2016). Deep Learning, Chapter 11: Practical Methodology. MIT Press.*
- *Ludwig 0.2.1 visualization documentation. GitHub: ludwig-ai/ludwig, tag v0.2.1.*

In [ ]:
# Check which results directories were created by the training runs
if os.path.isdir('results'):
    for entry in sorted(os.listdir('results')):
        print(entry)
else:
    print('No results directory found - run at least one training cell first.')

In [ ]:
# Update run_dir to match the directory printed above
run_dir = 'results/api_experiment_run'  # adjust if Ludwig used a different name
plots_dir = os.path.join(run_dir, 'plots')
os.makedirs(plots_dir, exist_ok=True)

# Generate learning curves - verified against Ludwig 0.2.1 CLI flags
!ludwig visualize \
    --visualization learning_curves \
    --training_statistics {run_dir}/training_statistics.json \
    -od {plots_dir} \
    -ff png

In [ ]:
# Display accuracy learning curve
accuracy_plot = os.path.join(plots_dir, 'learning_curves_label_accuracy.png')
loss_plot     = os.path.join(plots_dir, 'learning_curves_combined_loss.png')

if os.path.isfile(accuracy_plot):
    display(im(accuracy_plot))
else:
    print(f'Plot not found at {accuracy_plot} - check that the visualization cell ran successfully.')

if os.path.isfile(loss_plot):
    display(im(loss_plot))
else:
    print(f'Plot not found at {loss_plot}')

In [ ]:
# Evaluate the most recently trained model on the held-out test set
# model_cnnrnn is trained last; swap for model_rnn or model_cnn to compare
predictions, stats = model_cnnrnn.test(data_df=test_df)

test_df['prediction'] = predictions['label_predictions']
accuracy = sum(test_df['label'].apply(str) == test_df['prediction']) / len(test_df)
print(f'CNNRNN test accuracy: {accuracy:.4f}')
test_df.head()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->

> **After examining the learning curves from each model run: which model shows the clearest sign of overfitting? In the cell below, record the model name, the specific feature of its learning curves that indicates overfitting, and one change to the Ludwig configuration that would address it.**

<br>

```python
# Model with clearest overfitting signal:
# Evidence from learning curves (e.g., 'validation loss rises after epoch X while training loss keeps falling'):
# Proposed fix in Ludwig config (e.g., reduce early_stop, add dropout, change encoder):
```

<hr style="border: 2px solid#003262;" />

In [ ]:
# Model with clearest overfitting signal:
# Evidence from learning curves:
# Proposed fix in Ludwig config:


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## WRAP UP AND NEXT STEPS

In this tutorial you applied two foundational regularization techniques - early stopping and learning rate control - to manage overfitting in Ludwig NLP classifiers, and used encoder architecture comparison as a practical cross-validation strategy to identify which inductive bias best fits the Stanford Sentiment Treebank.

The core ideas transfer to any declarative ML setup:

- **Regularization** constrains how complex a function the model can fit. Early stopping does this by capping effective training steps; dropout and weight decay do it by penalizing model parameters directly. All three are levers worth tuning together.
- **Cross-validation**, in its practical form, means testing more than one configuration on held-out data before committing to a final model. Even a fixed train/val/test split with multiple architectures gives more reliable generalization estimates than a single model selected without comparison.

<br>

**Next steps:**

- Add `dropout` to the encoder configuration in Ludwig (e.g., `'fc_dropout': 0.3` in the input feature config) and observe the effect on the validation/training loss gap.
- Try the BERT encoder (see Appendix for setup details). Transformer encoders are typically fine-tuned rather than trained from scratch, which changes both the learning rate regime and the early stopping behavior.
- Formalize the cross-validation by running each encoder across multiple random seeds and averaging test accuracy across runs to get a more reliable estimate of each architecture's generalization performance.

<hr style="border: 6px solid#003262;" />

<a id="Appendix_1"></a>

<hr style="border: 2px solid#003262;" />

#### APPENDIX

## **INSTALLATION** AND SETUP

This notebook uses Ludwig 0.2.1, which was built on TensorFlow 1.15. The TF 2.x API (installed by default via `pip install tensorflow`) is not compatible - the cell below uninstalls TF 2.x and installs the TF 1.15 GPU build.

**Requirements:** Python 3.6-3.7, CUDA-capable GPU strongly recommended (training CNNRNN and BERT without GPU is very slow).

**To install from requirements.txt:**

```bash
python3 -m venv venv
source venv/bin/activate
pip install -r requirements.txt
```

**BERT encoder (optional):** To use the BERT encoder cell (not included in the main tutorial above), download the BERT-tiny checkpoint from the [Google Research BERT repository](https://github.com/google-research/bert) and update the `config_path`, `checkpoint_path`, and `vocab_file` paths in the model definition. See the Ludwig 0.2.1 documentation for the full encoder configuration.

___

<strong style="color:red">KEY CONSIDERATION:</strong> Running the cells below will modify your Python environment. Run them in an isolated virtual environment, not in your base environment.

___

In [ ]:
# Install Ludwig 0.2.1
!pip install ludwig==0.2.1

In [ ]:
# Ludwig 0.2.1 requires TensorFlow 1.15, not TF 2.x
!pip uninstall tensorflow -y
!pip install tensorflow-gpu==1.15.2  # replace with tensorflow==1.15.2 for CPU-only

In [ ]:
!pip install torchtext==0.6.0 nltk tqdm